# F1: Corpus Construction
## Overview

This project builds a digital analytical edition of the prose and poetry of Jonathan Swift, drawn from Project Gutenberg. The goal of the F0 to F1 transformation was to convert raw source files into a structured corpus suitable for downstream text analytics, with consistent indexing and metadata across heterogeneous works.

## Corpus Scope

**Final corpus**: 49 distinct works, 3,585 paragraphs, drawn from 7 Project Gutenberg source files.

**Source files included**:

| PG # | File | Works produced |
|---|---|---|
| 1080 | A Modest Proposal | 1 |
| 4738 | Three Prayers and Sermons | 6 |
| 623 | The Battle of the Books and Other Short Pieces | 18 |
| 37156 | Ireland in the Days of Dean Swift | 15 |
| 60186 | Polite Conversation in Three Dialogues | 4 |
| 4737 | A Tale of a Tub | 4 |
| 829 | Gulliver's Travels | 1 |

## Methodological Approach

### Defining the F1 Unit (Segmentation Policy)

Because the included works range widely in length and structure, treating each source file as a single document would have caused longer works to dominate downstream analyses. A per-work segmentation policy was therefore established:

- **Each chapter** of *Gulliver's Travels*
- **Each section** of *A Tale of a Tub*
- **Each individual short piece** in the *Battle of the Books* bundle
- **Each sermon and prayer** in *Three Prayers and Sermons*
- **Each tract** in *Ireland in the Days of Dean Swift*
- **Each dialogue** in *Polite Conversation in Three Dialogues*
- **The whole essay** for *A Modest Proposal*

### OCHO Index Design

A unified Ordered Content Hierarchy of Objects (OCHO) was designed to accommodate works with varying internal structure:

```
work_id > part_num > chap_num > para_num > sent_num > token_num
```

Works with shallow structure (e.g., *A Modest Proposal*) pad `part_num` and `chap_num` to 1, while works with deeper structure (*Gulliver's Travels*: 4 parts x 8-12 chapters) populate them meaningfully. A boolean `has_chapter_structure` field in the LIBRARY table flags works whose internal divisions are analytically meaningful, enabling downstream filtering for chapter-level analyses.

### Parser Architecture

Rather than writing a single monolithic parser, the project utilized a small collection of helpers and progressively more sophisticated parsing strategies. Claude was used as a collaborator to identify per-work metadata (genres, publication years, modes) and validate parsing decisions against the structure of each source files.

**Shared helpers** (used by all parsers):

- `read_gutenberg_file()` — reads a Gutenberg .txt file into a line-indexed DataFrame
- `find_pattern_line()` — returns the line index of the first regex match in a DataFrame
- `slice_between_headers()` — splits a DataFrame into chunks bounded by header lines
- `trim_to_content()` — slices a line DataFrame to just Swift's content using regex anchors
- `split_into_paragraphs()` — splits content lines into cleaned paragraph strings, filtering decorative dividers and stray Roman numerals
- `build_paragraph_records()` — converts a paragraph list into OCHO-indexed records
- `make_library_row()` — builds a consistent LIBRARY row dict, ensuring uniform schema across all works

**Three parsing patterns**, in increasing structural complexity:

1. **Single-work pattern** (`acquire_a_modest_proposal`): one source file, one work, flat sequence of paragraphs. Used for self-contained essays.

2. **Multi-work bundle pattern** (`acquire_multi_work_bundle`): one source file, many independent works, each treated as flat. Configured per-file via a `works_metadata` list specifying boundary regexes, titles, and genre labels for each contained work. The pattern includes a `'skip': True` option to use a section as a boundary marker without emitting it. Used for the *Battle of the Books* bundle, *Three Prayers and Sermons*, *Ireland in the Days of Dean Swift*, and *Polite Conversation*.

3. **Structured single-work patterns** — two variants:
   - `acquire_structured_single_work`: one work with one level of internal sections, populating `chap_num` across paragraphs. Used for *A Tale of a Tub*.
   - `acquire_structured_with_parts`: one work with two levels of nested hierarchy, populating both `part_num` and `chap_num`. Used for *Gulliver's Travels*.

### Boundary Detection Strategy

Boundaries between works (in bundle files) and between sections (in structured works) were detected via per-work compiled regex patterns matching specific header strings in the source. Several recurring complications were handled:

- **Wrapped headers**: Gutenberg's plaintext format wraps lines at ~70 characters, so multi-word headers often span multiple lines. Regexes were anchored on a one-line fragment unique to each header.
- **Table-of-contents collisions**: many headers appear twice in source files: once in a TOC and once as the actual section. Discriminators (case sensitivity, leading whitespace, trailing periods) were used to match only the actual section.
- **Repeated chapter numbering**: in *Gulliver's Travels*, "CHAPTER I." appears four times (once per part). The chapter regex is applied within each part's sliced subset rather than globally, ensuring chapters are numbered correctly within their part.

### Editorial Stripping

Most source files include substantial editorial framing (introductions, footnotes, transcriber notes) by editors such as Henry Morley, J. Bowles Daly, and George Saintsbury. These were excluded from the F1 corpus by trimming each file to start at Swift's first authorial content and end before any editorial endmatter. Where editorial content is interleaved with Swift's (e.g., footnotes after sermons in pg4738), the trim anchors were chosen to exclude it.

### Metadata Schema

Each work in the LIBRARY table carries the following fields:

- `work_id` — short, readable identifier (e.g., `gullivers_travels`, `prayer_for_stella_2`)
- `source_file` — original Gutenberg filename for provenance
- `pg_number` — Gutenberg eBook number
- `title` — full title of the work
- `year_written` — approximate composition year for time-based analyses
- `genre` — fine-grained genre label (e.g., `satirical_essay`, `prayer`, `political_tract`, `birthday_poem`)
- `mode` — rhetorical stance (`ironic`, `sincere`, or `mixed`)
- `parsing_notes` — provenance and any parsing decisions specific to that work
- `has_chapter_structure` — boolean flag for works with analytically meaningful internal hierarchy

Genre labels were assigned at fine granularity (20 distinct values across the corpus) to maximize downstream analytical flexibility.

## Corpus Statistics

**Genre distribution** (selected):

- Political tracts: 11 works (Drapier's Letters, Irish economic pamphlets)
- Birthday poems: 5 works (the Stella poems)
- Sermons: 3 works
- Prayers: 3 works
- Satirical dialogues: 3 works (Polite Conversation)
- Satirical essays: 3 works
- Verse satires: 2 works
- Plus singletons in 10 additional categories

**Structural depth**:

- 47 works with flat structure (`chap_num = 1` throughout)
- *A Tale of a Tub* with 14 internal sections
- *Gulliver's Travels* with 39 chapters across 4 parts

**Paragraph density per part of *Gulliver's Travels***:

- Part I (Lilliput): 128 paragraphs
- Part II (Brobdingnag): 119 paragraphs
- Part III (Laputa): 174 paragraphs
- Part IV (Houyhnhnms): 195 paragraphs

The increasing density across the four voyages aligns with scholarly understanding that Swift's later voyages contain his most discursive philosophical material.

## Tools and Acknowledgments

The corpus was built in Python using `pandas`, `re`, and standard file-handling libraries. Claude was used as a collaborator for identifying per-work metadata (genre classification, publication years, rhetorical mode), and validating parsing decisions against source-file structure. All code was authored, tested, and integrated by the project author.

## Setup

In [35]:
import pandas as pd
import numpy as np
import re
import os

In [36]:
OHCO = ['work_id','part_num','chap_num','para_num','sent_num','token_num']

F0_path = "data/F0_sources"

## Helper Functions        

In [37]:
def read_gutenberg_file(F0_path, source_file):
    """
    Read a Gutenberg .txt file into a line-indexed DataFrame.
    
    Returns:
    df: DataFrame with one row per line indexed by line_num with a single column 'line_str'.
    """
    file_path = f"{F0_path}/{source_file}"
    lines = open(file_path, 'r', encoding='utf-8-sig').readlines()
    df = pd.DataFrame(lines, columns=['line_str'])
    df.index.name = 'line_num'
    df.line_str = df.line_str.str.strip()
    return df

def find_pattern_line(df, pat, label, source_file):
    """Return the line index of the first match for `pat` in df, or raise."""
    matches = df['line_str'].str.contains(pat.pattern, regex=True, flags=pat.flags, na=False)
    if not matches.any():
        raise ValueError(f"Could not find {label} in {source_file}")
    return matches.idxmax()

def slice_between_headers(df, header_lines, end_line=None):
    """
    Slice `df` into chunks bounded by header_lines. Each header line itself is excluded.
    `end_line` is the boundary for the final chunk (defaults to df.index.max() + 1).
    Returns a list of DataFrames, one per header_line.
    """
    if end_line is None:
        end_line = df.index.max() + 1
    chunks = []
    for i, line in enumerate(header_lines):
        chunk_start = line + 1
        chunk_end = header_lines[i + 1] if i + 1 < len(header_lines) else end_line
        chunks.append(df.loc[chunk_start:chunk_end - 1])
    return chunks

def trim_to_content(df, start_pat, end_pat):
    """
    Slice the line DataFrame to just the content between two regex anchors.
    start_pat: included in result. end_pat: excluded from result.
    """
    matches_start = df['line_str'].str.contains(start_pat.pattern, regex=True, flags=start_pat.flags, na=False)
    matches_end = df['line_str'].str.contains(end_pat.pattern, regex=True, flags=end_pat.flags, na=False)
    return df.loc[matches_start.idxmax():matches_end.idxmax() - 1]

def clean_editorial_apparatus(text):
    """
    Strip Project Gutenberg editorial apparatus from text.
    """
    prev = None
    while prev != text:
        prev = text
        text = re.sub(r'\[[^\[\]]*\]', '', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text

def split_into_paragraphs(df):
    """
    Convert a line-level DataFrame into a list of paragraph strings.
    
    Args:
    df: line DataFrame
    
    Returns:
    list of cleaned paragraph strings.
    """
    full_text = '\n'.join(df['line_str'].tolist())
    full_text = clean_editorial_apparatus(full_text)
    raw_paragraphs = re.split(r'\n\s*\n+', full_text)
    
    paragraphs_list = []
    for p in raw_paragraphs:
        cleaned = re.sub(r'\s+', ' ', p).strip()
        if not cleaned:
            continue
        if re.fullmatch(r'[\*\s]+', cleaned):
            continue
        if re.fullmatch(r'[IVX]+\.?\s*', cleaned, re.IGNORECASE):
            continue
        paragraphs_list.append(cleaned)
    
    return paragraphs_list

def build_paragraph_records(paragraphs_list, work_id, part_num=1, chap_num=1):
    """
    Turn a list of paragraph strings into OCHO-indexed records.
    
    Args:
    paragraphs_list: list of paragraph strings
    work_id: the work's ID
    part_num: OCHO level
    chap_num: OCHO level
    
    Returns:
    list of dicts ready to be appended to a master paragraphs list.
    """
    records = []
    for i, para_str in enumerate(paragraphs_list, start=1):
        records.append({
            'work_id':  work_id,
            'part_num': part_num,
            'chap_num': chap_num,
            'para_num': i,
            'para_str': para_str,
        })
    return records

def make_library_row(work_id, source_file, pg_number, title, year_written,
                     genre, mode, parsing_notes='', has_chapter_structure=False):
    """
    Build a single LIBRARY row dict with consistent keys across all parsers.
    """
    return { 
        'work_id': work_id,
        'source_file': source_file,
        'pg_number': pg_number,
        'title': title,
        'year_written': year_written,
        'genre': genre,
        'mode': mode,
        'parsing_notes': parsing_notes,
        'has_chapter_structure': has_chapter_structure,
    }

def acquire_multi_work_bundle(F0_path, source_file, pg_number,
                              start_pat, end_pat,
                              works_metadata, OHCO=OHCO):
    """Generic parser for any Gutenberg file containing multiple distinct works."""
    df = read_gutenberg_file(F0_path, source_file)
    df = trim_to_content(df, start_pat, end_pat)
    
    header_lines = [
        find_pattern_line(df, meta['boundary_pat'], f"boundary for {meta['work_id']}", source_file)
        for meta in works_metadata
    ]
    chunks = slice_between_headers(df, header_lines)
    
    all_paragraphs = []
    library_rows = []
    for meta, chunk_df in zip(works_metadata, chunks):
        if meta.get('skip', False):
            continue
        paragraphs_list = split_into_paragraphs(chunk_df)
        all_paragraphs.extend(build_paragraph_records(paragraphs_list, meta['work_id']))
        library_rows.append(make_library_row(
            work_id=meta['work_id'],
            source_file=source_file,
            pg_number=pg_number,
            title=meta['title'],
            year_written=meta['year_written'],
            genre=meta['genre'],
            mode=meta['mode'],
            parsing_notes=meta.get('parsing_notes', f'extracted from {source_file} multi-work bundle'),
            has_chapter_structure=meta.get('has_chapter_structure', False),
        ))
    
    paragraphs = pd.DataFrame(all_paragraphs).set_index(OHCO[:4])[['para_str']]
    return library_rows, paragraphs

def acquire_structured_single_work(F0_path, source_file, pg_number, work_meta,
                                   start_pat, end_pat, section_boundaries, OHCO=OHCO):
    """Parser for a single work with internal section structure (one level deep)."""
    df = read_gutenberg_file(F0_path, source_file)
    df = trim_to_content(df, start_pat, end_pat)
    
    header_lines = [
        find_pattern_line(df, pat, f"section '{label}'", source_file)
        for label, pat in section_boundaries
    ]
    chunks = slice_between_headers(df, header_lines)
    
    work_id = work_meta['work_id']
    all_paragraphs = []
    for chap_num, chunk_df in enumerate(chunks, start=1):
        paragraphs_list = split_into_paragraphs(chunk_df)
        for para_num, para_str in enumerate(paragraphs_list, start=1):
            all_paragraphs.append({
                'work_id': work_id,
                'part_num': 1,
                'chap_num': chap_num,
                'para_num': para_num,
                'para_str': para_str,
            })
    
    paragraphs = pd.DataFrame(all_paragraphs).set_index(OHCO[:4])[['para_str']]
    library_row = make_library_row(
        work_id=work_meta['work_id'],
        source_file=source_file,
        pg_number=pg_number,
        title=work_meta['title'],
        year_written=work_meta['year_written'],
        genre=work_meta['genre'],
        mode=work_meta['mode'],
        parsing_notes=work_meta.get('parsing_notes', f'structured single work from {source_file}'),
        has_chapter_structure=True,
    )
    return [library_row], paragraphs

def acquire_structured_with_parts(F0_path, source_file, pg_number, work_meta,
                                  start_pat, end_pat,
                                  part_boundaries, chapter_pat, OHCO=OHCO):
    """Parser for a single work with two-level hierarchy: Parts containing Chapters."""
    df = read_gutenberg_file(F0_path, source_file)
    df = trim_to_content(df, start_pat, end_pat)
    
    part_header_lines = [
        find_pattern_line(df, pat, f"part '{label}'", source_file)
        for label, pat in part_boundaries
    ]
    part_chunks = slice_between_headers(df, part_header_lines)
    
    work_id = work_meta['work_id']
    all_paragraphs = []
    
    for part_num, part_df in enumerate(part_chunks, start=1):
        chapter_matches = part_df['line_str'].str.contains(
            chapter_pat.pattern, regex=True, flags=chapter_pat.flags, na=False
        )
        chapter_lines = part_df.index[chapter_matches].tolist()
        chap_chunks = slice_between_headers(part_df, chapter_lines, end_line=part_df.index.max() + 1)
        
        for chap_num, chap_df in enumerate(chap_chunks, start=1):
            paragraphs_list = split_into_paragraphs(chap_df)
            for para_num, para_str in enumerate(paragraphs_list, start=1):
                all_paragraphs.append({
                    'work_id': work_id,
                    'part_num': part_num,
                    'chap_num': chap_num,
                    'para_num': para_num,
                    'para_str': para_str,
                })
    
    paragraphs = pd.DataFrame(all_paragraphs).set_index(OHCO[:4])[['para_str']]
    library_row = make_library_row(
        work_id=work_meta['work_id'],
        source_file=source_file,
        pg_number=pg_number,
        title=work_meta['title'],
        year_written=work_meta['year_written'],
        genre=work_meta['genre'],
        mode=work_meta['mode'],
        parsing_notes=work_meta.get('parsing_notes', f'structured single work with parts from {source_file}'),
        has_chapter_structure=True,
    )
    return [library_row], paragraphs

## Parser Functions

In [38]:
def acquire_a_modest_proposal(F0_path=F0_path, OHCO=OHCO):
    work_id = 'modest_proposal'
    source_file = 'pg1080_A_Modest_Proposal.txt'
    pg_number = 1080
    title = 'A Modest Proposal'
    
    df = read_gutenberg_file(F0_path, source_file)
    
    start_pat = re.compile(r'^It is a melancholy object', re.IGNORECASE)
    end_pat = re.compile(r'\*\*\*\s*END OF (?:THE |THIS )?PROJECT GUTENBERG', re.IGNORECASE)
    df = trim_to_content(df, start_pat, end_pat)
    
    paragraphs_list = split_into_paragraphs(df)
    
    para_records = build_paragraph_records(paragraphs_list, work_id)
    paragraphs = pd.DataFrame(para_records).set_index(OHCO[:4])[['para_str']]
    
    library_row = make_library_row(
        work_id = work_id,
        source_file = source_file,
        pg_number = pg_number,
        title = title,
        year_written = 1729,
        genre = 'satirical_essay',
        mode = 'ironic',
        parsing_notes = 'single document, paragraphs only; no internal divisions',
    )
    
    return [library_row], paragraphs

In [39]:
def acquire_three_prayers_and_sermons(F0_path=F0_path, OHCO=OHCO):
    works_metadata = [
        {
            'work_id':'prayer_for_stella_1',
            'title':'First Prayer for Stella',
            'year_written': 1727,
            'genre':'prayer',
            'mode':'sincere',
            'boundary_pat': re.compile(r'^\s*IN HER LAST SICKNESS,\s*1727', re.IGNORECASE),
        },
        {
            'work_id':'prayer_for_stella_2',
            'title':'Second Prayer for Stella (Oct 17, 1727)',
            'year_written': 1727,
            'genre':'prayer',
            'mode':'sincere',
            'boundary_pat': re.compile(r'^\s*WRITTEN OCTOBER 17,\s*1727'),
        },
        {
            'work_id': 'prayer_for_stella_3',
            'title': 'Third Prayer for Stella (Nov 6, 1727)',
            'year_written': 1727,
            'genre': 'prayer',
            'mode': 'sincere',
            'boundary_pat': re.compile(r'^\s*WRITTEN NOVEMBER 6,\s*1727'),
        },
        {
            'work_id': 'sermon_mutual_subjection',
            'title': 'On Mutual Subjection',
            'year_written': 1744,
            'genre': 'sermon',
            'mode': 'sincere',
            'boundary_pat': re.compile(r'^\s*ON MUTUAL SUBJECTION'),
        },
        {
            'work_id': 'sermon_sleeping_in_church',
            'title': 'On Sleeping in Church',
            'year_written': 1744,
            'genre': 'sermon',
            'mode': 'sincere',
            'boundary_pat': re.compile(r'^\s*ON SLEEPING IN CHURCH'),
        },
        {
            'work_id': 'sermon_wisdom_of_this_world',
            'title': 'On the Wisdom of This World',
            'year_written': 1744,
            'genre': 'sermon',
            'mode': 'sincere',
            'boundary_pat': re.compile(r'^\s*ON THE WISDOM OF THIS WORLD'),
        },
    ]
    
    return acquire_multi_work_bundle(
        F0_path = F0_path,
        source_file = 'pg4738_Three_Prayers_and_Sermons.txt',
        pg_number = 4738,
        start_pat = re.compile(r'^\s*USED BY THE DEAN FOR STELLA', re.IGNORECASE),
        end_pat = re.compile(r'^\s*FOOTNOTES\.\s*$'),
        works_metadata = works_metadata,
        OHCO = OHCO,
    )

In [40]:
def acquire_battle_of_books_bundle(F0_path=F0_path, OHCO=OHCO):    
    works_metadata = [
        {
            'work_id': 'battle_of_books',
            'title': 'The Battle of the Books',
            'year_written': 1697,
            'genre':'allegorical_satire',
            'mode': 'ironic',
            'boundary_pat': re.compile(r'^\s*THE BOOKSELLER TO THE READER\.?\s*$'),
            'parsing_notes':'includes Bookseller prefatory note and Author preface; main satire begins after',
        },
        {
            'work_id': 'meditation_upon_broomstick',
            'title': 'A Meditation Upon a Broomstick',
            'year_written': 1701,
            'genre': 'parody',
            'mode': 'ironic',
            'boundary_pat': re.compile(r'^\s*A MEDITATION UPON A BROOMSTICK\.?\s*$'),
        },
        {
            'work_id': 'predictions_for_1708',
            'title': 'Predictions for the Year 1708',
            'year_written': 1708,
            'genre': 'mock_prediction',
            'mode': 'ironic',
            'boundary_pat': re.compile(r'^\s*PREDICTIONS FOR THE YEAR 1708\.?\s*$'),
        },
        {
            'work_id': 'accomplishment_of_predictions',
            'title': "The Accomplishment of the First of Mr. Bickerstaff's Predictions",
            'year_written': 1708,
            'genre': 'mock_report',
            'mode': 'ironic',
            'boundary_pat': re.compile(r'^\s*THE ACCOMPLISHMENT OF THE FIRST OF MR\.'),
        },
        {
            'work_id':'baucis_and_philemon',
            'title':'Baucis and Philemon',
            'year_written': 1709,
            'genre':'mythological_poem',
            'mode':'mixed',
            'boundary_pat': re.compile(r'^\s*BAUCIS AND PHILEMON\.?\s*$'),
        },
        {
            'work_id': 'logicians_refuted',
            'title': 'The Logicians Refuted',
            'year_written': 1707,
            'genre': 'verse_satire',
            'mode': 'ironic',
            'boundary_pat': re.compile(r'^\s*THE LOGICIANS REFUTED\.?\s*$'),
        },
        {
            'work_id': 'puppet_show',
            'title': 'The Puppet Show',
            'year_written': 1721,
            'genre': 'verse_satire',
            'mode': 'ironic',
            'boundary_pat': re.compile(r'^\s*THE PUPPET SHOW\.?\s*$'),
        },
        {
            'work_id': 'cadenus_and_vanessa',
            'title': 'Cadenus and Vanessa',
            'year_written': 1713,
            'genre': 'autobiographical_poem',
            'mode': 'mixed',
            'boundary_pat': re.compile(r'^\s*CADENUS AND VANESSA\.?\s*$'),
        },
        {
            'work_id': 'stella_birthday_1718',
            'title': "Stella's Birthday, 1718",
            'year_written': 1718,
            'genre': 'birthday_poem',
            'mode': 'sincere',
            'boundary_pat': re.compile(r"^\s*STELLA'S BIRTHDAY,\s*1718\.?\s*$"),
        },
        {
            'work_id': 'stella_birthday_1720',
            'title':  "Stella's Birthday, 1720",
            'year_written': 1720,
            'genre': 'birthday_poem',
            'mode': 'sincere',
            'boundary_pat': re.compile(r"^\s*STELLA'S BIRTHDAY,\s*1720\.?\s*$"),
        },
        {
            'work_id': 'stella_birthday_1722',
            'title': "Stella's Birthday, 1722",
            'year_written': 1722,
            'genre': 'birthday_poem',
            'mode': 'sincere',
            'boundary_pat': re.compile(r"^\s*STELLA'S BIRTHDAY\.?\s*$"),
        },
        {
            'work_id': 'stella_birthday_1724',
            'title': "Stella's Birthday, 1724",
            'year_written': 1724,
            'genre':  'birthday_poem',
            'mode': 'sincere',
            'boundary_pat': re.compile(r"^\s*STELLA'S BIRTHDAY,\s*1724\.?\s*$"),
        },
        {
            'work_id':'stella_birthday_1726',
            'title':"Stella's Birthday, March 13, 1726",
            'year_written': 1726,
            'genre': 'birthday_poem',
            'mode': 'sincere',
            'boundary_pat': re.compile(r"^\s*STELLA'S BIRTHDAY,\s*MARCH 13,\s*1726\.?\s*$"),
        },
        {
            'work_id': 'to_stella_visiting_sickness',
            'title': 'To Stella, Visiting me in my sickness',
            'year_written': 1727,
            'genre': 'personal_poem',
            'mode': 'sincere',
            'boundary_pat': re.compile(r"^\s*TO STELLA,\s*$"),
        },
        {
            'work_id':'SKIP_prayer_oct_1727',
            'title':'(skipped duplicate)',
            'year_written': 1727,
            'genre':'prayer',
            'mode':'sincere',
            'boundary_pat': re.compile(r'^\s*THE FIRST HE WROTE OCT\.\s*17,\s*1727\.?\s*$'),
            'skip': True,
        },
        {
            'work_id': 'SKIP_prayer_nov_1727',
            'title':'(skipped duplicate)',
            'year_written': 1727,
            'genre': 'prayer',
            'mode': 'sincere',
            'boundary_pat': re.compile(r'^\s*THE SECOND PRAYER WAS WRITTEN NOV\.\s*6,\s*1727\.?\s*$'),
            'skip': True,
        },
        {
            'work_id': 'beasts_confession',
            'title': "The Beasts' Confession",
            'year_written': 1732,
            'genre': 'beast_fable',
            'mode': 'ironic',
            'boundary_pat': re.compile(r"^\s*THE BEASTS' CONFESSION"),
        },
        {
            'work_id': 'argument_abolishing_christianity',
            'title':'An Argument Against Abolishing Christianity',
            'year_written': 1708,
            'genre':'satirical_essay',
            'mode':'ironic',
            'boundary_pat': re.compile(r'^\s*AN ARGUMENT TO PROVE THAT THE ABOLISHING'),
        },
        {
            'work_id':'hints_essay_conversation',
            'title':'Hints Towards an Essay on Conversation',
            'year_written': 1710,
            'genre': 'social_essay',
            'mode': 'sincere',
            'boundary_pat': re.compile(r'^\s*HINTS TOWARDS AN ESSAY ON CONVERSATION\.?\s*$'),
        },
        {
            'work_id':'thoughts_various_subjects',
            'title': 'Thoughts on Various Subjects',
            'year_written': 1727,
            'genre': 'aphorisms',
            'mode':'mixed',
            'boundary_pat': re.compile(r'^\s*THOUGHTS ON VARIOUS SUBJECTS\.?\s*$'),
        },
    ]
    
    return acquire_multi_work_bundle(
        F0_path = F0_path,
        source_file = 'pg623_Battle_of_the_Books_and_Other_Short_Pieces.txt',
        pg_number= 623,
        start_pat = re.compile(r'^\s*THE BOOKSELLER TO THE READER\.?\s*$'),
        end_pat = re.compile(r'^\s*FOOTNOTES:\s*$'),
        works_metadata = works_metadata,
        OHCO = OHCO,
    )

In [41]:
def acquire_ireland_tracts(F0_path=F0_path, OHCO=OHCO):
    works_metadata = [
        {
            'work_id': 'drapiers_letter_1',
            'title': "The Drapier's First Letter (To the Shop-keepers, Tradesmen, Farmers, and Common People of Ireland)",
            'year_written': 1724,
            'genre': 'political_tract',
            'mode': 'ironic',
            'boundary_pat': re.compile(r'^\s*BRETHREN, FRIENDS, COUNTRYMEN'),
        },
        {
            'work_id': 'drapiers_letter_2',
            'title': "The Drapier's Second Letter",
            'year_written': 1724,
            'genre': 'political_tract',
            'mode': 'ironic',
            'boundary_pat': re.compile(r'^\s*SECOND LETTER\.?\s*$'),
        },
        {
            'work_id': 'drapiers_letter_3',
            'title': "The Drapier's Third Letter",
            'year_written': 1724,
            'genre': 'political_tract',
            'mode': 'ironic',
            'boundary_pat': re.compile(r'^\s*THIRD LETTER\.?\s*$'),
        },
        {
            'work_id': 'drapiers_letter_4',
            'title': "The Drapier's Fourth Letter (To the Whole People of Ireland)",
            'year_written': 1724,
            'genre': 'political_tract',
            'mode': 'ironic',
            'boundary_pat': re.compile(r'^\s*FOURTH LETTER\.?\s*$'),
        },
        {
            'work_id': 'drapiers_letter_5',
            'title': "The Drapier's Fifth Letter",
            'year_written': 1724,
            'genre': 'political_tract',
            'mode': 'ironic',
            'boundary_pat': re.compile(r'^\s*THE FIFTH LETTER\s*$'),
        },
        {
            'work_id': 'drapiers_letter_6',
            'title': "The Drapier's Sixth Letter",
            'year_written': 1724,
            'genre': 'political_tract',
            'mode': 'ironic',
            'boundary_pat': re.compile(r'^\s*SIXTH LETTER\s*$'),
        },
        {
            'work_id': 'drapiers_letter_7',
            'title': "The Drapier's Seventh Letter",
            'year_written': 1724,
            'genre': 'political_tract',
            'mode': 'ironic',
            'boundary_pat': re.compile(r'^\s*SEVENTH LETTER\s*$'),
        },
        {
            'work_id': 'address_to_jury',
            'title': 'The Address to the Jury',
            'year_written': 1724,
            'genre': 'political_tract',
            'mode': 'mixed',
            'boundary_pat': re.compile(r'^\s*THE ADDRESS TO THE JURY\.?\s*$'),
        },
        {
            'work_id': 'description_of_quilca',
            'title': "Swift's Description of Quilca",
            'year_written': 1725,
            'genre': 'satirical_sketch',
            'mode': 'ironic',
            'boundary_pat': re.compile(r"^\s*SWIFT'S DESCRIPTION OF QUILCA\.?\s*$"),
        },
        {
            'work_id': 'blunders_of_quilca',
            'title': 'The Blunders, Deficiencies, Distresses, and Misfortunes of Quilca',
            'year_written': 1725,
            'genre': 'satirical_sketch',
            'mode': 'ironic',
            'boundary_pat': re.compile(r'^\s*THE BLUNDERS, DEFICIENCIES'),
        },
        {
            'work_id': 'answer_to_a_paper',
            'title': 'Answer to a Paper',
            'year_written': 1728,
            'genre': 'political_tract',
            'mode': 'ironic',
            'boundary_pat': re.compile(r'^\s*ANSWER TO A PAPER,\s*$'),
        },
        {
            'work_id': 'maxims_controlled',
            'title': 'Maxims Controulled in Ireland',
            'year_written': 1729,
            'genre': 'political_tract',
            'mode': 'ironic',
            'boundary_pat': re.compile(r'^\s*MAXIMS CONTROULED IN IRELAND\.?\s*$'),
        },
        {
            'work_id': 'injured_lady',
            'title': 'The Story of the Injured Lady',
            'year_written': 1707,
            'genre': 'allegorical_tract',
            'mode': 'ironic',
            'boundary_pat': re.compile(r'^\s*THE STORY OF THE INJURED LADY\.?\s*$'),
        },
        {
            'work_id': 'answer_to_injured_lady',
            'title': 'The Answer to the Injured Lady',
            'year_written': 1707,
            'genre': 'allegorical_tract',
            'mode': 'ironic',
            'boundary_pat': re.compile(r'^\s*THE ANSWER TO THE INJURED LADY\.?\s*$'),
        },
        {
            'work_id': 'two_letters_irish_improvement',
            'title': 'Two Letters on Subjects Relative to the Improvement of Ireland',
            'year_written': 1729,
            'genre': 'political_tract',
            'mode': 'sincere',
            'boundary_pat': re.compile(r'^\s*TO MESSRS\. TRUMAN AND LAYFIELD\.?\s*$'),
        },
    ]
    
    return acquire_multi_work_bundle(
        F0_path=F0_path,
        source_file='pg37156_Ireland_in_the_Days_of_Dean_Swift.txt',
        pg_number=37156,
        start_pat=re.compile(r"^\s*THE DRAPIER'S LETTERS\.?\s*$"),
        end_pat=re.compile(r'\*\*\*\s*END OF (?:THE |THIS )?PROJECT GUTENBERG', re.IGNORECASE),
        works_metadata=works_metadata,
        OHCO=OHCO,
    )

In [42]:
def acquire_polite_conversation(F0_path=F0_path, OHCO=OHCO):
    works_metadata = [
        {
            'work_id': 'polite_conversation_intro',
            'title': 'Polite Conversation: Introduction',
            'year_written': 1738,
            'genre': 'satirical_essay',
            'mode': 'ironic',
            'boundary_pat': re.compile(r'^\s*AN INTRODUCTION TO THE FOLLOWING TREATISE\.?\s*$'),
        },
        {
            'work_id': 'polite_conversation_dialogue_1',
            'title': 'Polite Conversation: Dialogue I',
            'year_written': 1738,
            'genre': 'satirical_dialogue',
            'mode': 'ironic',
            'boundary_pat': re.compile(r'^\s*POLITE CONVERSATION, ETC\.\s*$'),
        },
        {
            'work_id': 'polite_conversation_dialogue_2',
            'title': 'Polite Conversation: Dialogue II',
            'year_written': 1738,
            'genre': 'satirical_dialogue',
            'mode': 'ironic',
            'boundary_pat': re.compile(r'^\s*DIALOGUE II\.?\s*$'),
        },
        {
            'work_id': 'polite_conversation_dialogue_3',
            'title': 'Polite Conversation: Dialogue III',
            'year_written': 1738,
            'genre': 'satirical_dialogue',
            'mode': 'ironic',
            'boundary_pat': re.compile(r'^\s*DIALOGUE III\.?\s*$'),
        },
    ]
    
    return acquire_multi_work_bundle(
        F0_path=F0_path,
        source_file='pg60186_Polite_Conversation_in_Three_Dialogues.txt',
        pg_number=60186,
        start_pat=re.compile(r'^\s*AN INTRODUCTION TO THE FOLLOWING TREATISE\.?\s*$'),
        end_pat=re.compile(r'^\s*ILLUSTRATIVE NOTES\.?\s*$'),
        works_metadata=works_metadata,
        OHCO=OHCO,
    )

In [43]:
def acquire_tale_of_a_tub(F0_path=F0_path, OHCO=OHCO):
    source_file = 'pg4737_A_Tale_of_a_Tub.txt'
    pg_number = 4737
    
    tale_of_a_tub_meta = {
        'work_id': 'tale_of_a_tub',
        'title': 'A Tale of a Tub',
        'year_written': 1704,
        'genre': 'allegorical_satire',
        'mode': 'ironic',
        'parsing_notes': 'sectioned: Epistle Dedicatory, Preface, Sections I-XI, Conclusion (14 sections total)',
    }
    
    roman_numerals = ['I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII', 'IX', 'X', 'XI']

    section_boundaries = (
        [('epistle_dedicatory', re.compile(r'^\s*THE EPISTLE DEDICATORY\s*$')),
         ('preface', re.compile(r'^\s*THE PREFACE\.?\s*$'))]
        + [(f'section_{i+1}', re.compile(rf'^\s*SECTION {n}\.?\s*$'))
           for i, n in enumerate(roman_numerals)]
        + [('conclusion', re.compile(r'^\s*THE CONCLUSION\.?\s*$'))]
    )
    
    tale_lib, tale_paras = acquire_structured_single_work(
        F0_path=F0_path,
        source_file=source_file,
        pg_number=pg_number,
        work_meta=tale_of_a_tub_meta,
        start_pat=re.compile(r'^\s*THE EPISTLE DEDICATORY\s*$'),
        end_pat=re.compile(r'^THE HISTORY OF MARTIN\.\s*$'),
        section_boundaries=section_boundaries,
        OHCO=OHCO,
    )
    
    companion_works_metadata = [
        {
            'work_id': 'history_of_martin',
            'title': 'The History of Martin',
            'year_written': 1704,
            'genre': 'allegorical_satire',
            'mode': 'ironic',
            'boundary_pat': re.compile(r'^THE HISTORY OF MARTIN\.\s*$'),
        },
        {
            'work_id': 'digression_on_wars',
            'title': 'A Digression on the Nature, Usefulness, and Necessity of Wars and Quarrels',
            'year_written': 1704,
            'genre': 'satirical_essay',
            'mode': 'ironic',
            # Title wraps onto two lines in the file; anchor on the first
            'boundary_pat': re.compile(r'^\s*A DIGRESSION ON THE NATURE, USEFULNESS'),
        },
        {
            'work_id': 'project_universal_benefit',
            'title': 'A Project for the Universal Benefit of Mankind',
            'year_written': 1704,
            'genre': 'satirical_essay',
            'mode': 'ironic',
            'boundary_pat': re.compile(r'^\s*A PROJECT FOR THE UNIVERSAL BENEFIT OF MANKIND\.?\s*$'),
        },
    ]
    
    companion_lib, companion_paras = acquire_multi_work_bundle(
        F0_path=F0_path,
        source_file=source_file,
        pg_number=pg_number,
        start_pat=re.compile(r'^\s*THE HISTORY OF MARTIN\.?\s*$'),
        end_pat=re.compile(r'^\s*FOOTNOTES\.?\s*$'),
        works_metadata=companion_works_metadata,
        OHCO=OHCO,
    )
    
    library_rows = tale_lib + companion_lib
    paragraphs = pd.concat([tale_paras, companion_paras])
    
    return library_rows, paragraphs

In [44]:
def acquire_gullivers_travels(F0_path=F0_path, OHCO=OHCO):
    work_meta = {
        'work_id': 'gullivers_travels',
        'title': "Gulliver's Travels",
        'year_written': 1726,
        'genre': 'satirical_fiction',
        'mode': 'ironic',
        'parsing_notes': '4 parts (voyages) x 8-12 chapters each; 39 chapters total',
    }
    
    part_boundaries = [
        ('part_1_lilliput',    re.compile(r'^PART I\. A VOYAGE TO LILLIPUT')),
        ('part_2_brobdingnag', re.compile(r'^PART II\. A VOYAGE TO BROBDINGNAG')),
        ('part_3_laputa',      re.compile(r'^PART III\. A VOYAGE TO LAPUTA')),
        ('part_4_houyhnhnms',  re.compile(r'^PART IV\. A VOYAGE TO THE COUNTRY OF THE HOUYHNHNMS')),
    ]

    chapter_pat = re.compile(r'^CHAPTER [IVX]+\.\s*$')
    
    return acquire_structured_with_parts(
        F0_path=F0_path,
        source_file='pg829_Gullivers_Travels.txt',
        pg_number=829,
        work_meta=work_meta,
        start_pat=re.compile(r'^PART I\. A VOYAGE TO LILLIPUT'),
        end_pat=re.compile(r'\*\*\*\s*END OF (?:THE |THIS )?PROJECT GUTENBERG', re.IGNORECASE),
        part_boundaries=part_boundaries,
        chapter_pat=chapter_pat,
        OHCO=OHCO,
    )

## Parse and Combine

In [45]:
F1_path = "data/F1"
os.makedirs(F1_path, exist_ok=True)

parsers = [
    ('A Modest Proposal', acquire_a_modest_proposal),
    ('Three Prayers and Sermons', acquire_three_prayers_and_sermons),
    ('Battle of the Books bundle', acquire_battle_of_books_bundle),
    ('Ireland tracts', acquire_ireland_tracts),
    ('Polite Conversation', acquire_polite_conversation),
    ('Tale of a Tub', acquire_tale_of_a_tub),
    ("Gulliver's Travels", acquire_gullivers_travels),
]

lib_rows_all = []
doc_dfs_all = []

for name, parser_func in parsers:
    print(f"Running {name}...", end=" \n")
    try:
        lib_rows, docs = parser_func()
        lib_rows_all.extend(lib_rows)
        doc_dfs_all.append(docs)
        print(f"{len(lib_rows)} works, {len(docs)} paragraphs")
    except Exception as e:
        print(f"Failed: {e}")
        raise

LIB = pd.DataFrame(lib_rows_all).set_index('work_id')
DOC = pd.concat(doc_dfs_all)

print("\n\nCORPUS SUMMARY")
print(f"Total works: {len(LIB)}")
print(f"Total paragraphs: {len(DOC)}")
print(f"Unique genres: {LIB['genre'].nunique()}")
print(f"Year range: {LIB['year_written'].min()} – {LIB['year_written'].max()}")
print(f"Source files: {LIB['source_file'].nunique()}")
print(f"Structured works: {LIB['has_chapter_structure'].sum()}")

print("\nGenre distribution:")
print(LIB['genre'].value_counts().to_string())

lib_csv_path = os.path.join(F1_path, 'LIB.csv')
doc_csv_path = os.path.join(F1_path, 'DOC.csv')

LIB.to_csv(lib_csv_path)
DOC.to_csv(doc_csv_path)

Running A Modest Proposal... 
1 works, 33 paragraphs
Running Three Prayers and Sermons... 
6 works, 93 paragraphs
Running Battle of the Books bundle... 
18 works, 386 paragraphs
Running Ireland tracts... 
15 works, 596 paragraphs
Running Polite Conversation... 
4 works, 1601 paragraphs
Running Tale of a Tub... 
4 works, 251 paragraphs
Running Gulliver's Travels... 
1 works, 619 paragraphs


CORPUS SUMMARY
Total works: 49
Total paragraphs: 3579
Unique genres: 20
Year range: 1697 – 1744
Source files: 7
Structured works: 2

Genre distribution:
genre
political_tract          11
satirical_essay           5
birthday_poem             5
allegorical_satire        3
satirical_dialogue        3
sermon                    3
prayer                    3
allegorical_tract         2
satirical_sketch          2
verse_satire              2
social_essay              1
aphorisms                 1
personal_poem             1
beast_fable               1
parody                    1
autobiographical_poem     1